# Sensor Ranking by Levels of Activity over Time

## Reading and Parsing Relevant Data

### Reading in elevation data from USGS

The next few lines of code are intended to pull USGS elevation data from USGS' API endpoint and merge it with the FloodNet's "final_deployed_sensors.geojson" dataset to attach elevation data to each sensor's location.

In [35]:
import os                                    # check whether the elevation cache file already exists
import pandas as pd                          # tabular data for the elevation cache
import geopandas as gpd                      # read the sensor locations from geojson with geometry support
import requests                              # call the USGS elevation API
from requests.adapters import HTTPAdapter    # lets us attach a retry policy to a requests.Session
from urllib3.util.retry import Retry         # retry/backoff policy for rate-limit and server errors

FORCE_REFRESH = False    # set True to ignore the cache and re-fetch every sensor's elevation

In [36]:
USGS_EPQS = "https://epqs.nationalmap.gov/v1/json"    # USGS Elevation Point Query Service endpoint


def _usgs_epqs_session():
    # Session with automatic retries on rate-limit (429) and server-side (5xx) errors,
    # using exponential backoff so repeated failures don't hammer the API.
    s = requests.Session()
    retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])  # 3 tries, waits grow 2s/4s/8s
    s.mount("https://", HTTPAdapter(max_retries=retry))    # attach the retry policy to all HTTPS requests on this session
    return s


def fetch_elevation_ft(lat, lon, session):
    r = session.get(USGS_EPQS, params={
        "x": lon,                  # USGS expects longitude as x
        "y": lat,                  # and latitude as y
        "units": "Feet",           # elevation comes back in feet
        "wkid": 4326,              # coordinate system code for standard lat/lon (WGS84)
        "includeDate": "false",    # we only need the value, not the raster's survey date
    }, timeout=30)                 # give up after 30s rather than hang indefinitely
    r.raise_for_status()           # raise on a bad HTTP status instead of silently returning junk
    return r.json()["value"]       # "value" is the elevation at this point, in the requested units

In [37]:
SENSORS_PATH = "../final_deployed_sensors.geojson"    # shared sensor table one level up
ELEVATION_CACHE = r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\4_sensor_activity\sensor_elevation.csv"    # cache scoped to this project only

sensors_gdf = gpd.read_file(SENSORS_PATH)    # load all 415 sensors with their lat/long

if not FORCE_REFRESH and os.path.exists(ELEVATION_CACHE):
    elevation_df = pd.read_csv(ELEVATION_CACHE)    # skip the API entirely if we already fetched this
    print(f"Loaded cached elevations from {ELEVATION_CACHE}")
else:
    session = _usgs_epqs_session()
    rows = []
    total = len(sensors_gdf)
    for i, row in enumerate(sensors_gdf.itertuples(), start=1):    # itertuples is faster than iterrows at this scale
        elev_ft = fetch_elevation_ft(row.latitude, row.longitude, session)    # one API call per sensor
        rows.append({"sensor_id": row.sensor_id, "elevation_ft": elev_ft})
        if i % 10 == 0 or i == total:    # progress print so a long run doesn't look frozen
            print(f"{i}/{total} sensors fetched")
    elevation_df = pd.DataFrame(rows)
    elevation_df.to_csv(ELEVATION_CACHE, index=False)    # save so this never has to run again unless FORCE_REFRESH is True
    print(f"Fetched and cached elevations for {len(elevation_df)} sensors")

elevation_df.head()

Loaded cached elevations from C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\4_sensor_activity\sensor_elevation.csv


,sensor_id,elevation_ft
0,BK-livonia-ave-georgia-ave-2tzr8o,24.343846
1,BK-lott-ave-thatford-ave-2tzqlc,17.322911
2,BK-vermont-st-linden-blvd-2tzqx0,14.829428
3,BK-halsey-st-saratoga-ave-2tzmno,41.174618
4,BK-saint-marks-ave-brooklyn-ave-2tzmc0,71.850439


In [38]:
sensors_gdf = sensors_gdf.merge(elevation_df, on="sensor_id")    # attach elevation_ft onto each sensor's row

# Sanity check: lowest-elevation sensors should skew toward coastal boroughs/waterfront streets
print(sensors_gdf["elevation_ft"].describe())    # quick range/spread check for implausible values
sensors_gdf.sort_values("elevation_ft")[["sensor_name", "borough", "elevation_ft"]].head(10)    # eyeball the 10 lowest

count    415.000000
mean      26.076973
std       26.533740
min        1.312364
25%        7.611605
50%       16.010523
75%       34.104349
max      170.603743
Name: elevation_ft, dtype: float64


,sensor_name,borough,elevation_ft
207,SI - Grimsby St/ Mapleton Ave,Staten Island,1.312364
192,SI - Quincy Ave/Iona St,Staten Island,1.640456
385,SI - Baden Pl/ Mapleton Ave,Staten Island,1.837310
297,SI - Olympia Blvd/ Mapleton Ave,Staten Island,2.198191
298,Q - Church Rd/E 6th Rd,Queens,2.821547
348,Q - Brookville Blvd/ Snake Rd 3,Queens,2.887203
209,Q - Beach 84 St (2),Queens,3.084030
395,Q - Brookville Blvd/ Snake Rd 2,Queens,3.084034
253,SI - Hylan Blvd/ Jefferson Blvd,Staten Island,3.116906
273,Q - Brookville Blvd/ Snake Rd 1,Queens,3.149703


#### Visualization: Elevation Gradient Map

In [39]:
import numpy as np
import folium
import branca.colormap as bcm

# Single-hue sequential ramp -- reversed so lowest elevation renders darkest, matching flood-risk intuition
BLUE_SEQUENTIAL = [
    "#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec",
    "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab",
    "#184f95", "#104281", "#0d366b",
]
REVERSED_BLUE = BLUE_SEQUENTIAL[::-1]

elev_min = sensors_gdf["elevation_ft"].min()
elev_max = sensors_gdf["elevation_ft"].max()

# A few high outliers dominate a linear scale, compressing the low end where most sensors
# (and the flood-relevant differences) actually sit. GAMMA > 1 packs more color breakpoints
# into the low end and fewer into the high end -- raise it for a stronger effect.
GAMMA = 3
color_positions = np.linspace(0, 1, len(REVERSED_BLUE)) ** GAMMA
nonlinear_index = elev_min + (elev_max - elev_min) * color_positions

elevation_colormap = bcm.LinearColormap(
    colors=REVERSED_BLUE,
    index=nonlinear_index,    # non-uniform breakpoints -- this is what makes the scale nonlinear
    vmin=elev_min,
    vmax=elev_max,
    caption="Sensor elevation (ft)",
)

elevation_map = folium.Map(
    location=[sensors_gdf["latitude"].mean(), sensors_gdf["longitude"].mean()],
    zoom_start=11,
    tiles="cartodbpositron",    # plain, low-clutter basemap
)

for row in sensors_gdf.itertuples():
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=5,
        color=elevation_colormap(row.elevation_ft),
        fill=True,
        fill_color=elevation_colormap(row.elevation_ft),
        fill_opacity=0.85,
        weight=1,
        tooltip=f"{row.sensor_name} — {row.elevation_ft:.1f} ft",    # hover shows the exact value per sensor
    ).add_to(elevation_map)

elevation_colormap.add_to(elevation_map)    # legend, so color isn't the only way to read the value
elevation_map

In [40]:
display(elevation_df.loc[elevation_df['elevation_ft'].idxmax()])
display(elevation_df.loc[elevation_df['elevation_ft'].idxmin()])

sensor_id       BX-e-208th-st-bainbridge-ave-2calqo
elevation_ft                             170.603743
Name: 94, dtype: object

sensor_id       SI-grimsby-st-mapleton-ave-1de5w0
elevation_ft                             1.312364
Name: 207, dtype: object

### Building the distance matrix

This section involves choosing an overall radius that will be used to determine neighboring sensor locations and test whether those neighboring sensors show similar data patterns.

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

DIST_CRS = "EPSG:2263"    # NAD83 / New York Long Island (ftUS) -- local projected CRS, so distances come out in feet

sensors_ft = sensors_gdf.to_crs(DIST_CRS)    # reproject: flat Euclidean math in feet instead of lat/long degrees
coords = np.column_stack([sensors_ft.geometry.x, sensors_ft.geometry.y])    # (415, 2) array of projected coordinates
sensor_ids = sensors_ft["sensor_id"].values

nn_model = NearestNeighbors(metric="euclidean")    # scikit-learn's kNN index, fit once and reused for every query below
nn_model.fit(coords)

In [ ]:
# How many sensors would have zero neighbors at each candidate radius? radius_neighbors returns
# every point within the radius INCLUDING the point itself, so len(...) - 1 is the true neighbor count.
for radius_ft in [500, 1000, 1500, 2000]:
    neighbor_lists = nn_model.radius_neighbors(coords, radius=radius_ft, return_distance=False)
    n_no_neighbor = sum(1 for neighbors in neighbor_lists if len(neighbors) <= 1)
    print(f"radius={radius_ft} ft: {n_no_neighbor}/{len(sensor_ids)} sensors have no neighbor within radius")

In [ ]:
# kneighbors returns each point's own closest matches, including itself at rank 0 (distance 0) --
# n_neighbors=6 gets self plus the 1st through 5th nearest OTHER sensors.
distances, _ = nn_model.kneighbors(coords, n_neighbors=6)

for k, label in [(1, "1st nearest"), (2, "2nd nearest"), (3, "3rd nearest"), (5, "5th nearest")]:
    d = distances[:, k]
    print(f"{label}: median={np.median(d):.0f} ft, 90th pct={np.percentile(d, 90):.0f} ft, max={d.max():.0f} ft")

**Why k=1 (not k=2) as the primary neighbor:** k=2's own distance profile confirms k=1 is the right primary choice. Median distance jumps from 1,487 ft (1st-nearest) to 2,346 ft (2nd-nearest) -- about 860 ft farther typically -- and the 2nd-nearest's 90th percentile (4,769 ft) already exceeds the 3,500 ft neighbor cap, meaning that for many sensors the 2nd-nearest wouldn't even qualify as "local" on its own. Using k=2 as the primary neighbor would systematically pick a farther, less-local comparison sensor for no benefit.

The 2nd-nearest still earns a place in `neighbors_df`, but only as a **fallback**, not a co-primary: specifically for the case where the 1st-nearest sensor is gap-tier (its silence can't be trusted as real evidence), not as a way to relax the distance requirement. See piece 3 above for that reasoning.

In [ ]:
NEIGHBOR_CAP_FT = 3500

# n_neighbors=3: rank 0 is the point itself, rank 1 is the nearest other sensor, rank 2 is the 2nd-nearest.
# kneighbors() always includes the query point as its own closest match (distance 0) because it's
# fit and queried on the same coords -- that's why n_neighbors=3 is needed to get 2 REAL neighbors.
distances, indices = nn_model.kneighbors(coords, n_neighbors=3)

# indices[:, 0] (self-match) is intentionally never touched -- columns 1 and 2 are the first
# and second real neighbors. indices holds positions into `coords`/`sensor_ids` (same row order,
# since sensor_ids was built from the same sensors_ft frame), so indexing sensor_ids by them
# turns "row 217 is nearest" into an actual sensor_id string.
nearest_idx, second_idx = indices[:, 1], indices[:, 2]
nearest_dist, second_dist = distances[:, 1], distances[:, 2]

neighbors_df = pd.DataFrame({
    "sensor_id": sensor_ids,
    "nearest_sensor_id": sensor_ids[nearest_idx],
    "nearest_dist_ft": nearest_dist,
    "second_nearest_sensor_id": sensor_ids[second_idx],
    "second_nearest_dist_ft": second_dist,
})
neighbors_df["has_neighbor"] = neighbors_df["nearest_dist_ft"] <= NEIGHBOR_CAP_FT    # False = "no comparable neighbor"

print(f"{neighbors_df['has_neighbor'].sum()}/{len(neighbors_df)} sensors have a comparable neighbor within {NEIGHBOR_CAP_FT} ft")
neighbors_df.head()

### Data refresh: continuous full-network NOAA tidal + 311 (through 2026-07-02)

The flood-plausible-day matching logic (a later step) needs rain, tide/surge, 311, and
neighbor-sensor signals for **every** day in the analysis window, across all 415 sensors --
not just the 128 QC-flagged dates and 11 sensors the existing `3_data_gap_exploration`
notebooks were built around. Those notebooks (`Tidal_Corroboration.ipynb`,
`311_Call_Verification.ipynb`) are finished analyses of the specific 2025 QC gap, so rather
than extending their date ranges in place (which wouldn't change their own conclusions),
this pulls fresh, continuous, full-network data scoped to this project -- same pattern
already used for elevation above.

- **NOAA observed water levels**: only the 2 NOAA stations (The Battery, Kings Point) need
  fetching here -- the other 4 tidal stations are USGS, whose observed levels + surge are
  already in `2_tidal_analysis/tidal_unified.geojson` through the full 2025-2026 window.
- **311 flooding calls**: same flood-relevant descriptor list `311_Call_Verification.ipynb`
  used, extended to the full 2025-01-01 to 2026-07-02 window, with an explicit `count(*)`
  check against the pull (Socrata silently truncates results without one -- bit us before
  on an unguarded query).

In [ ]:
NOAA_API = "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter"
NOAA_STATIONS = ["8518750", "8516945"]    # The Battery, Kings Point -- the 2 NOAA prediction-only stations
NOAA_START = "2025-01-01"
NOAA_END = "2026-07-02"
NOAA_TIDAL_CACHE = r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\4_sensor_activity\noaa_observed_water_levels.csv"


def _noaa_session():
    # Same retry/backoff pattern as the USGS elevation session above.
    s = requests.Session()
    retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s


def fetch_noaa_observed(station_id, start, end, session):
    # NOAA's API times out on long ranges, so fetch one calendar month at a time.
    recs = []
    for month_start in pd.date_range(start, end, freq="MS"):
        month_end = min(month_start + pd.offsets.MonthEnd(0), pd.Timestamp(end))
        r = session.get(NOAA_API, params={
            "product": "water_level",     # OBSERVED (not "predictions")
            "begin_date": month_start.strftime("%Y%m%d"),
            "end_date": month_end.strftime("%Y%m%d"),
            "datum": "MLLW", "station": station_id,
            "time_zone": "lst_ldt",       # local w/ DST -- converted to UTC below
            "interval": "6", "units": "english", "format": "json",
        }, timeout=60)
        r.raise_for_status()
        for row in r.json().get("data", []):
            if row.get("v") not in (None, ""):    # skip gaps (empty value)
                recs.append((station_id, row["t"], float(row["v"])))
    return recs


def mllw_above_navd(station_id):
    # How far MLLW sits above NAVD88 at this station (negative at NYC gauges) --
    # adding this to an MLLW height converts it to NAVD88, matching tidal_unified.geojson's datum.
    r = requests.get(
        f"https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/{station_id}/datums.json",
        params={"units": "english"}, timeout=30)
    r.raise_for_status()
    dd = {d["name"]: float(d["value"]) for d in r.json()["datums"]}
    return dd["MLLW"] - dd["NAVD88"]


if not FORCE_REFRESH and os.path.exists(NOAA_TIDAL_CACHE):
    noaa_obs = pd.read_csv(NOAA_TIDAL_CACHE, parse_dates=["datetime"])
    print(f"Loaded cached NOAA observed levels ({len(noaa_obs)} rows)")
else:
    session = _noaa_session()
    recs = [r for sid in NOAA_STATIONS for r in fetch_noaa_observed(sid, NOAA_START, NOAA_END, session)]
    noaa_obs = pd.DataFrame(recs, columns=["station_id", "t", "obs_mllw_ft"])
    # local wall-clock -> US/Eastern (resolve DST) -> UTC. NaT drops the spring-forward gap hour.
    noaa_obs["datetime"] = (pd.to_datetime(noaa_obs["t"])
                              .dt.tz_localize("US/Eastern", ambiguous="NaT", nonexistent="NaT")
                              .dt.tz_convert("UTC"))
    offsets = {sid: mllw_above_navd(sid) for sid in NOAA_STATIONS}
    noaa_obs["obs_navd_ft"] = noaa_obs["obs_mllw_ft"] + noaa_obs["station_id"].map(offsets)
    noaa_obs = noaa_obs.dropna(subset=["datetime"])[["station_id", "datetime", "obs_navd_ft"]]
    noaa_obs.to_csv(NOAA_TIDAL_CACHE, index=False)
    print(f"Fetched and cached {len(noaa_obs)} NOAA observed points ({NOAA_START} to {NOAA_END})")

noaa_obs.head()

In [ ]:
from urllib.parse import urlencode, quote

FLOOD_311_START = "2025-01-01T00:00:00"
FLOOD_311_END = "2026-07-02T23:45:00"
FLOOD_311_CACHE = r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\4_sensor_activity\flooding_311.csv"

# Same flood-relevant descriptor list used in 3_data_gap_exploration/4_311/311_Call_Verification.ipynb
FLOOD_DESCRIPTORS = [
    "Flooding on Highway", "Flooded", "Flooding on Street", "Street Flooding (SJ)",
    "Highway Flooding (SH)", "Ponding", "Puddle in Ground", "Puddle on Driveway",
    "Puddle on Sidewalk", "Catch Basin Clogged",
    "Catch Basin Clogged/Flooding (Use Comments) (SC)",
    "Manhole Overflow (Use Comments) (SA1)", "Backup",
    "Sewer Backup (Use Comments) (SA)", "Drain/Pipe Clogged", "Sewer or Drain",
    "Water or Sewer Runoff", "Water in Basement",
]


def soda_url(resource, **params):
    return f"https://data.cityofnewyork.us/resource/{resource}.json?" + urlencode(params, quote_via=quote)


descriptor_list = ", ".join(f'"{d}"' for d in FLOOD_DESCRIPTORS)
where_clause = (
    f'created_date BETWEEN "{FLOOD_311_START}" :: floating_timestamp '
    f'AND "{FLOOD_311_END}" :: floating_timestamp '
    f"AND caseless_one_of(descriptor, {descriptor_list})"
)
SELECT_COLS = "unique_key, created_date, descriptor, latitude, longitude, incident_address"

if not FORCE_REFRESH and os.path.exists(FLOOD_311_CACHE):
    flooding_311 = pd.read_csv(FLOOD_311_CACHE, parse_dates=["created_date"])
    print(f"Loaded cached {FLOOD_311_CACHE} ({len(flooding_311)} rows)")
else:
    # Explicit count(*) check -- Socrata silently truncates to 1,000 rows without $limit,
    # and even with $limit set, this confirms the pull isn't quietly short.
    total = int(pd.read_json(soda_url("erm2-nwe9", **{"$select": "count(*)", "$where": where_clause}))["count"][0])
    flooding_311 = pd.read_json(soda_url("erm2-nwe9", **{
        "$select": SELECT_COLS, "$where": where_clause,
        "$order": "created_date DESC NULL FIRST", "$limit": max(total, 1),
    }))
    assert len(flooding_311) == total, f"truncated: {len(flooding_311)} vs {total}"
    flooding_311.to_csv(FLOOD_311_CACHE, index=False)
    print(f"Fetched and cached {len(flooding_311)} 311 flooding reports "
          f"({FLOOD_311_START} to {FLOOD_311_END}, verified vs count(*))")

flooding_311.head()

### Gap-tier split

`floodnet_full_network_gap_tiers.csv` lists only the sensors that had a Tier 1-4 gap in
2025 (174 rows, not all 415) -- so the split has to be a join against the full sensor
list, not a read of the gap-tier file alone. Any sensor *not* in that file is treated as
non-gap-tier (clean) by default.

Gap-tier sensors get excluded from the main miss-rate ranking (a gap sensor's miss rate
reads ~100% almost by construction -- that's the gap itself, not evidence of
misplacement) and reported separately, heavily caveated. They still get a full local-risk
profile later (Tests A/B don't depend on a sensor's own feed data), just not the miss-rate
number.

In [ ]:
GAP_TIER_PATH = "../3_data_gap_exploration/1_gap_tiers/floodnet_full_network_gap_tiers.csv"


def norm_name(s):
    # strip + lowercase + collapse internal whitespace -- same normalization used
    # throughout this project's sensor-name reconciliation (handles stray spaces
    # like " M - Broad St/South St" without merging genuinely different sensors).
    return " ".join(str(s).strip().lower().split())


gap_tiers = pd.read_csv(GAP_TIER_PATH)
gap_tiers["name_key"] = gap_tiers["sensor_name"].apply(norm_name)
sensors_gdf["name_key"] = sensors_gdf["sensor_name"].apply(norm_name)

sensors_gdf = sensors_gdf.merge(
    gap_tiers[["name_key", "gap_tier"]], on="name_key", how="left"
)
sensors_gdf["is_gap_tier"] = sensors_gdf["gap_tier"].notna()

print(f"{sensors_gdf['is_gap_tier'].sum()}/{len(sensors_gdf)} sensors are gap-tier "
      f"(excluded from the main miss-rate ranking)")
print(f"{(~sensors_gdf['is_gap_tier']).sum()}/{len(sensors_gdf)} sensors are non-gap-tier "
      f"(get the full ranked treatment)")
sensors_gdf["gap_tier"].value_counts(dropna=False)